# PolyChain: Polymer Property Prediction
**AISEHack 2.0 - Polymer Property Prediction**
Targets: Tg (Glass Transition Temperature) & Egc (Chain Band Gap)

Run all cells in order. This notebook:
1. Installs dependencies
2. Loads competition data
3. Splits data by target type
4. Builds features (fingerprints + RDKit descriptors + polymer features)
5. Trains models per target (supports ridge, xgb, lgb, catboost, rf, mlp)
6. Blends ensembles per target
7. Merges submissions and generates submission.csv

In [ ]:
# Cell 1: Install dependencies (Kaggle: already have most)
import sys, os, json, pickle, subprocess, shutil, time
from pathlib import Path

try:
    import rdkit
except ImportError:
    print("Installing rdkit...")
    !pip install -q rdkit-pypi fastparquet

print("Python", sys.version[:6])
print("Dependencies ready")

In [ ]:
# Cell 2: Setup - detect environment, copy data
KAGGLE = Path("/kaggle").exists()
COLAB = Path("/content").exists() and not KAGGLE

if KAGGLE:
    DATA_SOURCE = Path("/kaggle/input/aisehack-2-0")
    WORK_DIR = Path("/kaggle/working/polymer_competition")
    sys.path.insert(0, str(WORK_DIR))
    os.chdir(WORK_DIR)
    shutil.copy(DATA_SOURCE / "train.csv", "data/train.csv")
    shutil.copy(DATA_SOURCE / "test.csv", "data/test.csv")
elif COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/Poly/polymer_competition")
    os.chdir(WORK_DIR)
    sys.path.insert(0, str(WORK_DIR))
    # Copy data from Drive
    src = Path("/content/drive/MyDrive/PolyChain/data")
    if src.exists():
        shutil.copy(src / "train.csv", "data/train.csv")
        shutil.copy(src / "test.csv", "data/test.csv")
else:
    WORK_DIR = Path.cwd()
    os.chdir(WORK_DIR)

import pandas as pd
import yaml
print(f"Environment: {'Kaggle' if KAGGLE else 'Colab' if COLAB else 'Local'}")
print(f"Work dir: {WORK_DIR}")

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
print(f"Train: {len(train)} rows")
print(f"Test: {len(test)} rows")

In [ ]:
# Cell 3: Split data by target type
from data.split_by_target import split_by_target
split_by_target("data/train.csv", "data/test.csv", "data/")

for t in ["tg", "egc"]:
    t_train = pd.read_csv(f"data/{t}/train.csv")
    t_test = pd.read_csv(f"data/{t}/test.csv")
    print(f"{t}: {len(t_train)} train, {len(t_test)} test")

In [ ]:
# Cell 4: Build features (fingerprints + descriptors + polymer features)
from features.build_features import build_features
build_features("config.yaml")

train_feat = pd.read_parquet("data/processed/features_train.parquet")
test_feat = pd.read_parquet("data/processed/features_test.parquet")
print(f"Features: train {train_feat.shape}, test {test_feat.shape}")

In [ ]:
# Cell 5: Generate CV splits per target
from data.generate_splits import generate_splits

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
N_FOLDS = cfg["cv"]["n_folds"]

for target in ["tg", "egc"]:
    generate_splits(
        f"data/{target}/train.csv",
        f"data/splits_{target}.pkl",
        n_folds=N_FOLDS, smiles_col="smiles", target_col="target",
    )

In [ ]:
# Cell 6: Train models per target per fold
# Models to train (add more: catboost, rf, mlp if time permits)
MODELS = ["ridge", "xgb", "lgb"]
TARGETS = ["tg", "egc"]

for target in TARGETS:
    print(f"\n{'='*50}")
    print(f"TARGET: {target}")
    print(f"{'='*50}")
    for model_type in MODELS:
        for fold in range(N_FOLDS):
            cmd = [
                sys.executable, "-m", "training.train",
                "--model_type", model_type,
                "--fold", str(fold),
                "--config", "config.yaml",
                "--target", target,
            ]
            t0 = time.time()
            result = subprocess.run(cmd, capture_output=True, text=True)
            elapsed = time.time() - t0
            if result.returncode == 0:
                # Extract metrics from stdout
                for line in result.stdout.split("\n"):
                    if "rmse" in line and "r2" in line:
                        print(f"  {model_type} fold {fold}: {line.strip()} ({elapsed:.0f}s)")
                        break
            else:
                print(f"  {model_type} fold {fold}: FAILED ({elapsed:.0f}s)")

In [ ]:
# Cell 7: Build ensemble per target
for target in TARGETS:
    result = subprocess.run([
        sys.executable, "-m", "ensemble.build_ensemble",
        "--config", "config.yaml", "--target", target,
    ], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(f"ERROR: {result.stderr}")

In [ ]:
# Cell 8: Merge per-target submissions into final submission.csv
result = subprocess.run([
    sys.executable, "-m", "data.merge_submissions",
    "--config", "config.yaml",
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(f"ERROR: {result.stderr}")

In [ ]:
# Cell 9: Validate and preview submission
sub = pd.read_csv("outputs/submissions/submission.csv")
print(f"Submission: {len(sub)} rows")
print(f"Columns: {list(sub.columns)}")
print(f"ID range: {sub['id'].min()} - {sub['id'].max()}")
print(f"\nFirst 10 rows:")
print(sub.head(10))
print(f"\nLast 5 rows:")
print(sub.tail(5))

# Copy to Kaggle output directory
if KAGGLE:
    shutil.copy("outputs/submissions/submission.csv", "/kaggle/working/submission.csv")
    print("\nCopied to /kaggle/working/submission.csv")

In [ ]:
# Cell 10: Summary report
print("="*60)
print("  PIPELINE COMPLETE")
print("="*60)
print(f"  Models trained: {len(MODELS)} x {len(TARGETS)} targets x {N_FOLDS} folds")
print(f"  Submission: outputs/submissions/submission.csv")
print(f"  Experiment log: experiments/manifest.json")
print("="*60)

# Show manifest summary
manifest_path = Path("experiments/manifest.json")
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    scores = [r for r in manifest if r.get("score") is not None]
    if scores:
        print(f"\n  Best R² scores:")
        df_scores = pd.DataFrame(scores)
        for target in TARGETS:
            t_scores = df_scores[df_scores["target"] == target]
            if len(t_scores):
                best = t_scores.loc[t_scores["score"].idxmax()]
                print(f"    {target}: {best['model']} fold {best['fold']} = {best['score']:.4f}")